# Hugging Face Fundamentals — Lesson 5: AutoModel

> Learning material for **Hugging Face Fundamentals**. Companion to the lesson script `05_AutoModel.py` (same content, runnable without Jupyter).

**Task ID:** HF-005  |  **Folder:** `05_AutoModel`


## What AutoModel loads

A pretrained model on the Hub is really a **folder** with a config and weights. `AutoModel.from_pretrained(id)` reads the config, figures out the architecture, downloads the weights, and builds the neural network for you.

> One function, thousands of architectures: BERT, GPT-2, T5, Llama, ...

**Step 1 — load DistilBERT** (a compressed BERT, ~66 M parameters, ~270 MB on disk — cached after the first download):


In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("distilbert-base-uncased")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")


**Step 2 — read the blueprint (config):**

In [ ]:
c = model.config
print("hidden size   :", c.hidden_size)     # width of every vector
print("layers        :", c.num_hidden_layers)
print("attention heads:", c.num_attention_heads)
print("vocab size    :", c.vocab_size)


## Step 3 — one forward pass

Give the model ids + attention mask, get a tensor of hidden states — **one vector per token**:


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
enc = tokenizer("Transformers are everywhere.", return_tensors="pt")
out = model(**enc)
print("output shape:", tuple(out.last_hidden_state.shape))
print("             (batch, tokens, hidden_size)")


## Step 4 — the task heads: AutoModelFor...

The plain `AutoModel` returns vectors only. For a real task you load a variant with a **head** on top — a small final layer that turns the vectors into scores:


In [ ]:
from transformers import AutoModelForSequenceClassification

head_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased"
)
logits = head_model(**enc).logits
print("logits shape:", tuple(logits.shape), " (batch, num_labels)")
print("num_labels  :", head_model.config.num_labels)


> The backbone stayed the same language model; only the head changed. That is why fine-tuning (module 2, HF-208) is so cheap — you retrain just the head.

## Try it yourself

1. Load `bert-base-uncased` and compare parameter counts (330 M vs 66 M).
2. Change the text — the output shape stays `(1, tokens, 768)`.
3. Try `AutoModelForQuestionAnswering` — inspect its heads in the config.

## Common pitfalls

- **Big downloads** — small models are fine on CPU; check sizes on the Hub.
- **Right class for the task** — `AutoModelForSequenceClassification` for classification, `AutoModelForSeq2SeqLM` for generation, ...
- **`UNEXPECTED`/`MISSING` keys in the load report** — normal when swapping heads; read them, don't panic.

## Summary

- AutoModel reads config → builds architecture → loads weights.
- One forward pass: `(batch, tokens, hidden_size)`.
- `AutoModelForTask` adds a task head on top of the backbone.

**Next lesson:** HF-006 — Model Inference.  |  Extra reading: `../resources/reference_links.md`
